# ReBRAC Broad Validation v2 — seed 43 supplement（N0 / N2' / N2'-asym，§5.8 补种子）

> 文档锚点：[`docs/rebrac_broad_validation_v2_seed43_supplement_plan.md`](../docs/rebrac_broad_validation_v2_seed43_supplement_plan.md)（**用户批准 2026-07-08，最小矩阵 +seed 43 × 3 单元**） · [`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) §6.2（预登记 3 seed） · [`docs/rebrac_broad_validation_v2_report.md`](../docs/rebrac_broad_validation_v2_report.md)（2-seed 基线 + §6.3 backlog）。

命令逐字克隆自 `rebrac_broad_validation_v2_core.ipynb`（N0/N2p）与 `rebrac_broad_validation_v2_n2p_asym_critic.ipynb`（asym），唯一新变量 = `--seed 43`。

## 既有 2-seed 基线（report §2 / §4.5）

| 单元 | seed 42 | seed 0 | verdict |
|---|---|---|---|
| N0（crosscomp / sub-critical） | 0.867 | 0.833 | HOLDS（≥0.70） |
| N2'（privileged / critical） | 0.000 | 0.000 | STRONG_NEGATIVE |
| N2'-asym（+asym critic） | 0.000 | 0.000 | ACTOR_FUNDAMENTAL_CONFIRMED |

## 预登记判读门槛（plan §2，跑前锁定、不允许 post-hoc 调整）

- **N0**：3-seed 均值 ≥ 0.70 且 < 0.902（vs anchor 同向退化）→ 一致；否则呈报。
- **N2'**：seed 43 = **0/30** → 一致；任何 >0 成功 → 呈报（「零成功完全一致」表述失效）。
- **N2'-asym**：seed 43 = **0/30** 且终止以越界为主（OOB 占多数）→ 一致；否则呈报。
- 三者全一致 → 仅做 plan §4 零论证定点微修；任一不一致 → 微修冻结，先呈报。

预算：3 × 64-epoch ReBRAC + 3 × 终检 ≈ **≤2h L4 单 session**。

## 0. 环境 sanity

In [ ]:
!nvidia-smi

## 1. Mount Drive + cwd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

## 2. Run matrix — 3 runs × seed 43

路径一律相对仓库根（Drive 路径含空格，`!python {var}` 插值不加引号会在空格处拆断）。

In [ ]:
import json
import os
from pathlib import Path

SEED = 43

# ---- 与首轮完全一致的两个 cell 定义（core notebook cell 1 逐字）----
N0_DATASET  = 'offline_data/crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz'
N0_MANIFEST = 'benchmarks/single_u10_cross_tgt15.json'
N2P_DATASET  = 'offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz'
N2P_MANIFEST = 'benchmarks/single_u15_cross_tgt15.json'

RUNS = [
    {'cell_id': 'N0',       'asym': False, 'dataset': N0_DATASET,  'manifest': N0_MANIFEST,
     'ckpt_dir': f'checkpoints/offline/rebrac/broad_validation_v2/N0/seed_{SEED}',
     'result_dir': f'results/offline/rebrac/broad_validation_v2/N0/seed_{SEED}',
     'baseline_dir': 'results/offline/rebrac/broad_validation_v2/N0',
     'regime_note': 'sub-critical (U=1.0/Re=150)'},
    {'cell_id': 'N2p',      'asym': False, 'dataset': N2P_DATASET, 'manifest': N2P_MANIFEST,
     'ckpt_dir': f'checkpoints/offline/rebrac/broad_validation_v2/N2p/seed_{SEED}',
     'result_dir': f'results/offline/rebrac/broad_validation_v2/N2p/seed_{SEED}',
     'baseline_dir': 'results/offline/rebrac/broad_validation_v2/N2p',
     'regime_note': 'critical (U=1.5/Re=250)'},
    {'cell_id': 'N2p_asym', 'asym': True,  'dataset': N2P_DATASET, 'manifest': N2P_MANIFEST,
     'ckpt_dir': f'checkpoints/offline/rebrac/broad_validation_v2_n2p_asym/seed_{SEED}',
     'result_dir': f'results/offline/rebrac/broad_validation_v2_n2p_asym/seed_{SEED}',
     'baseline_dir': 'results/offline/rebrac/broad_validation_v2_n2p_asym',
     'regime_note': 'critical (U=1.5/Re=250), asym critic'},
]

SUMMARIES_DIR = Path('results/offline/rebrac/broad_validation_v2/summaries')
VERDICT_JSON = SUMMARIES_DIR / 'seed43_supplement_verdict.json'

# eval 参数与首轮逐字一致（--episodes 100 受 manifest 30-ep 约束实跑 30）
EVAL_EPISODES    = 100
EVAL_SEED        = 123
EVAL_NUM_WORKERS = 4

os.environ['PYTHONUNBUFFERED'] = '1'

for i, r in enumerate(RUNS, 1):
    print(f"{i} {r['cell_id']:<9} asym={r['asym']!s:<5} ({r['regime_note']})")
    print(f"   ckpt:   {r['ckpt_dir']}")
    print(f"   result: {r['result_dir']}")

## 3. Preflight — dataset / manifest / 2-seed 基线就位

In [ ]:
for r in RUNS:
    for p in (r['dataset'], r['manifest']):
        if not Path(p).exists():
            raise FileNotFoundError(f"missing: {p} — sync repo/offline_data to Drive first")
print('[OK] datasets + manifests in place')

for r in RUNS:
    for s in (42, 0):
        f = Path(r['baseline_dir']) / f'seed_{s}' / 'test_result.json'
        if f.exists():
            v = float(json.loads(f.read_text(encoding='utf-8'))['eval_success_rate'])
            print(f"[OK] baseline {r['cell_id']} seed_{s}: success={v:.3f}")
        else:
            print(f"[WARN] baseline missing: {f}（§5 汇总将回退 report 转载值）")

## 4. Train — 3 runs × 64 epochs（`[skip]` 用 `agent_final.pt`）

vanilla 两单元与 asym 单元共用一条命令，`{EXTRA}` 只在 asym 追加 `--use-asymmetric-critic --privileged-actor-update-mode zeros`（与首轮 asym notebook 逐字一致）。

In [ ]:
import time

for r in RUNS:
    cdir = Path(r['ckpt_dir'])
    if (cdir / 'agent_final.pt').exists():
        print(f"[skip] {r['cell_id']} seed {SEED} done: {cdir}")
        continue
    cdir.mkdir(parents=True, exist_ok=True)

    dataset = r['dataset']
    manifest = r['manifest']
    save_dir = str(cdir)
    EXTRA = ('--use-asymmetric-critic --privileged-actor-update-mode zeros'
             if r['asym'] else '')

    print(f"\n========== train {r['cell_id']} seed={SEED} ({r['regime_note']}) ==========")
    t0 = time.time()
    !python -u -m scripts.train_offline \
        --algo rebrac \
        --offline-data {dataset} \
        --manifest {manifest} \
        --probe-layout s0 \
        --history-length 4 \
        --task-geometry cross_stream \
        --target-speed 1.5 \
        --objective arrival_v2 \
        --sampling-mode shuffle_no_replacement \
        --num-epochs 64 \
        --batch-size 256 \
        --hidden-dim 256 \
        --num-hidden-layers 3 \
        --actor-lr 3e-4 \
        --critic-lr 3e-4 \
        --gamma 0.99 \
        --tau 0.005 \
        --actor-penalty-coef 4.0 \
        --critic-penalty-coef 2.0 \
        --policy-noise 0.2 \
        --noise-clip 0.5 \
        --policy-freq 2 \
        --grad-clip-norm 10.0 \
        --normalizer-eps 1e-3 \
        --critic-layernorm \
        --no-actor-layernorm \
        --eval-every 0 \
        --skip-final-eval \
        --log-every 1000 \
        --seed {SEED} \
        --device cuda \
        --save-dir {save_dir} \
        {EXTRA}
    print(f"[done] {r['cell_id']} seed {SEED} ({(time.time()-t0)/60:.1f} min)")

## 5. Eval — 3 runs（命令与首轮逐字一致；`[skip]` 用 `test_result.json`）

In [ ]:
for r in RUNS:
    cdir = Path(r['ckpt_dir'])
    rdir = Path(r['result_dir'])
    test_json = rdir / 'test_result.json'
    if test_json.exists():
        print(f'[skip] eval done: {test_json}')
        continue
    if not (cdir / 'agent_final.pt').exists():
        print(f'[warn] missing agent_final.pt: {cdir}（训练未完成？）')
        continue
    rdir.mkdir(parents=True, exist_ok=True)

    ckpt = str(cdir)
    manifest = r['manifest']
    out_json = str(test_json)
    print(f"\n========== eval {r['cell_id']} seed={SEED} → {test_json} ==========")
    !python -u -m scripts.evaluate_offline \
        --checkpoint {ckpt} \
        --agent-file agent_final.pt \
        --manifest {manifest} \
        --episodes {EVAL_EPISODES} \
        --seed {EVAL_SEED} \
        --device cuda \
        --num-workers {EVAL_NUM_WORKERS} \
        --worker-device cpu \
        --output-json {out_json}

## 6. 汇总 + 预登记判读（plan §2 三门槛；输出 verdict JSON）

In [ ]:
import statistics

ANCHOR_EFF_V2_MEAN = 0.902  # main-line efficiency_v2 5-seed anchor

def read_success(path):
    d = json.loads(Path(path).read_text(encoding='utf-8'))
    n = int(float(d.get('num_eval_episodes', 0)))
    return float(d['eval_success_rate']), n, d.get('eval_termination_counts', {})

cells_out = {}
all_consistent = True
for r in RUNS:
    per_seed = {}
    for s in (42, 0, SEED):
        f = Path(r['baseline_dir']) / f'seed_{s}' / 'test_result.json'
        if f.exists():
            succ, n, counts = read_success(f)
            per_seed[str(s)] = {'success': succ, 'n': n, 'termination_counts': counts}
        else:
            per_seed[str(s)] = None
    vals = [v['success'] for v in per_seed.values() if v is not None]
    mean3 = statistics.mean(vals) if vals else float('nan')
    std3 = statistics.stdev(vals) if len(vals) >= 2 else float('nan')

    new = per_seed.get(str(SEED))
    if new is None:
        verdict = 'INCOMPLETE (seed 43 missing)'
        consistent = False
    elif r['cell_id'] == 'N0':
        consistent = (len(vals) == 3) and (0.70 <= mean3 < ANCHOR_EFF_V2_MEAN)
        verdict = 'CONSISTENT (HOLDS retained)' if consistent else 'ESCALATE'
    else:
        zero = (new['success'] == 0.0)
        if r['cell_id'] == 'N2p_asym':
            c = new['termination_counts']
            total = sum(c.values()) if c else 0
            oob_major = total > 0 and c.get('out_of_bounds', 0) > total / 2
            consistent = zero and oob_major
        else:
            consistent = zero
        verdict = 'CONSISTENT (zero-success intact)' if consistent else 'ESCALATE'
    all_consistent = all_consistent and consistent

    cells_out[r['cell_id']] = {'per_seed': per_seed, 'mean': mean3, 'std': std3,
                               'n_seeds': len(vals), 'verdict': verdict}
    print(f"{r['cell_id']:<9} per-seed " +
          ' '.join(f'{s}={v["success"]:.3f}' if v else f'{s}=NA' for s, v in per_seed.items()) +
          f'  mean={mean3:.3f} std={std3:.3f}  → {verdict}')

print('\nALL CONSISTENT:', all_consistent,
      '→ ' + ('可按 plan §4 零论证微修' if all_consistent else '微修冻结，先呈报'))

SUMMARIES_DIR.mkdir(parents=True, exist_ok=True)
VERDICT_JSON.write_text(json.dumps({
    'notebook': 'rebrac_broad_validation_v2_seed43_supplement',
    'seed_added': SEED,
    'preregistered_gates': 'plan §2: N0 mean3 in [0.70, 0.902); N2p seed43==0/30; '
                           'N2p_asym seed43==0/30 & OOB-majority',
    'cells': cells_out,
    'all_consistent': all_consistent,
}, indent=1, ensure_ascii=False), encoding='utf-8')
print(f'[saved] {VERDICT_JSON}')

## 7. 跑完后清单（回到 local）

```bash
# 在本地 repo root：只取回 results/（checkpoints 留 Drive）
rsync -av '<drive>/results/offline/rebrac/broad_validation_v2/' \
  results/offline/rebrac/broad_validation_v2/
rsync -av '<drive>/results/offline/rebrac/broad_validation_v2_n2p_asym/' \
  results/offline/rebrac/broad_validation_v2_n2p_asym/
```

回读判读入口：`paper/thesis_ch5/next_session_prompt.md`。
ALL CONSISTENT → plan §4 零论证微修（ground truth 先行，再 `boundary.tex` 定点 Edit + latexmk）；
任一 ESCALATE → 呈报硬停，§5.8 一字不动。